# ETL — Banco VDE (Ministério da Justiça)

Leitura e diagnóstico das planilhas `BancoVDE`.

In [ ]:
import polars as pl

In [ ]:
from pathlib import Path

from tcc.paths import DATA_DIR

dados = DATA_DIR / 'Dados MJ'
dfs = [
    pl.read_excel(f, infer_schema_length=None)
    for f in sorted(dados.glob('*.xlsx'))
]

df = pl.concat(dfs)
print(f'{len(dfs)} planilhas, {df.height} linhas')

In [ ]:
df.glimpse()

# Análise dos dados

In [ ]:
print('Valores ausentes por coluna (contagem e percentual):')

missing_counts = df.select(pl.all().is_null().sum())

total_rows = df.height
missing_percentage = df.select(
    ((pl.all().is_null().sum() / total_rows) * 100).round(2)
)

transposed_counts = missing_counts.transpose(include_header=True)
value_column_name_counts = transposed_counts.columns[1]
missing_df_transposed = transposed_counts.rename({'column': 'coluna', value_column_name_counts: 'valores_ausentes'})

transposed_percentage = missing_percentage.transpose(include_header=True)
value_column_name_perc = transposed_percentage.columns[1]
missing_perc_df_transposed = transposed_percentage.rename({'column': 'coluna', value_column_name_perc: 'percentual_ausente'})

combined_missing = missing_df_transposed.join(missing_perc_df_transposed, on='coluna', how='left')

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(combined_missing)

In [ ]:
print('\nValores únicos e contagens para a coluna `uf`:')
print(df.group_by('uf').len().sort('len', descending=True))

print('\nValores únicos e contagens para a coluna `evento`:')
print(df.group_by('evento').len().sort('len', descending=True))

print('\nValores únicos e contagens para a coluna `faixa_etaria`:')
print(df.group_by('faixa_etaria').len().sort('len', descending=True))

print('\nValores únicos e contagens para a coluna `abrangencia`:')
print(df.group_by('abrangencia').len().sort('len', descending=True))

In [ ]:
df

In [ ]:
# pl.Config.set_tbl_rows(-1)
df.select(pl.col("evento")).unique().show(limit=-1)